# 04 · EDA & Data Quality Analysis

**Purpose.** Understand the clean data before any model sees it: balance, patients, image properties, skin tone, and how easily the *source* can be guessed (shortcut risk).

| | |
|---|---|
| **Reads** | `data/processed/manifest_clean.csv` |
| **Writes** | `reports/figures/eda_*.png`, `reports/results/eda_summary.json` |
| **Run time (Colab T4)** | 2-3 min |
| **Next notebook** | 05 · Splits & Class Weights |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dermalens.labels import model_classes

TARGET_CLASSES = model_classes(open_set=CONFIG["data"]["open_set"])
from dermalens.utils import save_json

sns.set_theme(style="whitegrid")
data = pd.read_csv(paths.processed / "manifest_clean.csv", dtype={"phash": str})
findings = {}                                      # numbers quoted in the report

def save_figure(name):
    plt.tight_layout()
    plt.savefig(paths.figures / f"eda_{name}.png", dpi=150)
    plt.show()

print(f"{len(data):,} images | {data['group_id'].nunique():,} patient groups | {data['source'].nunique()} sources")

## 1. Class balance
A large gap between classes means the loss needs class weights (notebook 05).

In [ ]:
per_class = pd.crosstab(data["label"], data["source"]).reindex(TARGET_CLASSES)
per_class.plot.barh(stacked=True, figsize=(9, 4), title="Images per class, coloured by source")
plt.xlabel("images")
save_figure("class_balance")

totals = per_class.sum(axis=1)
findings["imbalance_ratio_max_over_min"] = round(float(totals.max() / totals.min()), 2)
print(totals.to_string())
print("Largest / smallest class:", findings["imbalance_ratio_max_over_min"])

## 2. Source vs class: the confounding check

If a class comes from only one dataset, the model can learn *"photos that look like dataset X"*
instead of the disease. 100% in one column below = that class and that source are inseparable.

In [ ]:
share = (pd.crosstab(data["label"], data["source"], normalize="index") * 100).reindex(TARGET_CLASSES).round(0)
plt.figure(figsize=(6, 4))
sns.heatmap(share, annot=True, fmt=".0f", cmap="Oranges", cbar_kws={"label": "% of class"})
plt.title("Where each class's images come from (%)")
save_figure("source_by_class")

single_source = share[(share == 100).any(axis=1)].index.tolist()
findings["single_source_classes"] = single_source
print("Classes from a single source:", single_source)

## 3. Photos per patient
Patients with many photos dominate if split carelessly; this is why notebook 05 splits by patient.

In [ ]:
per_patient = data.groupby(["source", "group_id"]).size().rename("photos").reset_index()
display(per_patient.groupby("source")["photos"].describe().round(1))
findings["max_photos_one_patient"] = int(per_patient["photos"].max())

per_patient[per_patient["source"] != "mcsi"].hist(column="photos", by="source", bins=20, figsize=(10, 3), sharey=True)
save_figure("photos_per_patient")

## 4. Image size by source
Different sizes per source are a clue a model could exploit; resizing every image to the same size removes it.

In [ ]:
display(data.groupby("source")[["width", "height"]].describe().round(0))
plt.figure(figsize=(7, 5))
sns.scatterplot(data=data, x="width", y="height", hue="source", alpha=0.5, s=15)
plt.xscale("log")
plt.yscale("log")
plt.title("Original image size (log scale)")
save_figure("image_size_by_source")

## 5. Colour, brightness and contrast
Do the sources look systematically different? Compared within the same class where possible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, ["brightness", "contrast", "mean_r"]):
    sns.boxplot(data=data, x="label", y=column, hue="source", ax=ax, order=TARGET_CLASSES, fliersize=1)
    ax.tick_params(axis="x", rotation=40)
    ax.set_title(column)
save_figure("colour_by_class_and_source")

## 6. How guessable is the source from simple statistics?

A plain logistic regression sees **only** size and average colour (no pixels, no lesion). If it can
name the source accurately, a deep network certainly can, and Healthy/Psoriasis could be "solved"
through the source. Notebook 11 repeats this test with a real CNN.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

simple_features = ["width", "height", "mean_r", "mean_g", "mean_b", "brightness", "contrast"]
X = np.column_stack([np.log(data["width"]), np.log(data["height"]), data[simple_features[2:]]])
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
folds = GroupKFold(n_splits=5)

source_acc = cross_val_score(model, X, data["source"], groups=data["group_id"], cv=folds).mean()
label_acc = cross_val_score(model, X, data["label"], groups=data["group_id"], cv=folds).mean()
majority = data["source"].value_counts(normalize=True).max()
findings["source_from_simple_stats_accuracy"] = round(float(source_acc), 3)
findings["label_from_simple_stats_accuracy"] = round(float(label_acc), 3)
print(f"Source predicted from size+colour only: {source_acc:.1%} (always guessing the biggest source: {majority:.1%})")
print(f"Label  predicted from size+colour only: {label_acc:.1%}")

## 7. Skin tone coverage
Only DermaCon-IN has Fitzpatrick (FST) and Monk (MST) labels; MCSI has a light/dark tag.

In [ ]:
derma = data[data["source"] == "dermaconin"]
fst_table = pd.crosstab(derma["label"], derma["fst"].astype("Int64"), margins=True)
display(fst_table)
display(data.loc[data["source"] == "mcsi", "tone_ld"].value_counts().rename("Healthy images").to_frame())
findings["images_with_fst"] = int(data["fst"].notna().sum())
findings["fst_counts"] = derma["fst"].value_counts().sort_index().to_dict()

derma["mst"].plot.hist(bins=range(1, 12), rwidth=0.9, figsize=(6, 3), title="Monk skin tone (DermaCon-IN)")
save_figure("monk_skin_tone")

## 8. Label confidence and demographics

In [ ]:
display(pd.crosstab(derma["label"], derma["label_confidence"]))
display(pd.crosstab(data["source"], data["sex"].fillna("unknown")))

## 9. Most common body sites per class (DermaCon-IN; one photo can list several)

In [ ]:
sites = derma.assign(site=derma["body_site"].str.split(",")).explode("site")
sites["site"] = sites["site"].str.strip()
display(sites.groupby("label")["site"].agg(lambda s: ", ".join(s.value_counts().head(4).index)).to_frame("top sites"))

## 10. Look at the data
Five random photos per class. Always look before training: label errors and odd images show up here.

In [ ]:
from dermalens.data import load_rgb

per_row = 5
fig, axes = plt.subplots(len(TARGET_CLASSES), per_row, figsize=(per_row * 2.4, len(TARGET_CLASSES) * 2.5))
for row, label in enumerate(TARGET_CLASSES):
    examples = data[data["label"] == label].sample(min(per_row, (data["label"] == label).sum()), random_state=CONFIG["seed"])
    for col in range(per_row):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(examples):
            item = examples.iloc[col]
            image = load_rgb(paths.raw / item["rel_path"])
            image.thumbnail((256, 256))
            ax.imshow(image)
            ax.set_title(f"{item['source']}", fontsize=8)
    axes[row, 0].text(-0.1, 0.5, label, transform=axes[row, 0].transAxes, ha="right", va="center", fontsize=10)
save_figure("sample_grid")

## 10b. Patient metadata: does it carry diagnostic signal? (v3)

If body region or age differs strongly between classes, giving the model this information should
raise accuracy. The heat maps show, for each class, the share of photos per body region and per
age group. A chi-square test checks whether the differences could be chance.

In [ ]:
from scipy.stats import chi2_contingency
from dermalens.metadata import BODY_REGIONS

regions = data.assign(region=data["meta_regions"].fillna("").str.split("|")).explode("region")
regions = regions[regions["region"].isin(list(BODY_REGIONS))]
region_share = (pd.crosstab(regions["label"], regions["region"], normalize="index") * 100).round(0)
age_share = (pd.crosstab(data["label"], data["meta_age"], normalize="index") * 100).round(0)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [2.2, 1]})
sns.heatmap(region_share, annot=True, fmt=".0f", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set_title("Body region by class (% of the class's photos)")
sns.heatmap(age_share, annot=True, fmt=".0f", cmap="Greens", ax=axes[1], cbar=False)
axes[1].set_title("Age group by class (%)")
save_figure("metadata_by_class")

for name, table in {"body region": pd.crosstab(regions["label"], regions["region"]),
                    "age group": pd.crosstab(data["label"], data["meta_age"]),
                    "sex": pd.crosstab(data["label"], data["meta_sex"])}.items():
    chi2, p_value, dof, _ = chi2_contingency(table)
    print(f"{name:<12} chi2 = {chi2:8.1f}  dof = {dof:3d}  p = {p_value:.2g}")
    findings[f"metadata_{name.replace(' ', '_')}_p_value"] = float(p_value)

A tiny p-value means the field differs between classes → worth feeding to the model (notebook 08 measures the actual gain).

## 11. Findings → decisions

In [ ]:
save_json(findings, paths.results / "eda_summary.json")
for key, value in findings.items():
    print(f"- {key}: {value}")

| If the numbers above show… | …then we |
|---|---|
| an imbalance ratio well above 1 | use inverse-frequency class weights (05) and report macro-F1, not accuracy |
| single-source classes | report results per source (09) and run the shortcut test (11) |
| sources guessable from size/colour | resize every image identically (06) and use random crops |
| many photos for some patients | split by patient group (05) |
| few images with FST 6 | report FST 3-4 vs 5-6 with confidence intervals (09) |